# Intro to Agentic AI

## Fall 2026

As I noted in class last week, we ended up a bit further ahead of where I thought we would be. The last time I taught this course, I added 3 sessions on AI, mostly as an introduction to how deep learning models are trained. This was essentially an introduction to the AI in Biology (BSC4892/6895) course that Arthur Porto and I teach in the Spring. If you want to check them out, the notebooks are [AI_01_Abalone_analysis.ipynb](AI_01_Abalone_analysis.ipynb), [AI_02_CNNs_part1.ipynb](AI_02_CNNs_part1.ipynb) and [AI_03_CNNs_part2.ipynb](AI_03_CNNs_part2.ipynb).

I started down that road with the [BioLab_Image_recognition.ipynb](BioLab_Image_recognition.ipynb) notebook, which was an attempt to bring more of the Python/SQL content into the notebook. Initially, I intended to follow that up with how the computer vision model works--and we may still end up there, and may use the CNN notebooks above (CNN is a convolutional neural network, one computer vision method)

But after some more thinking, I decided that rather than teaching how one aspect of AI (computer vision) works in limited detail--essentially trying to cram part of the AI in Biology course into this one--I would see how we can leverage AI in research. I think this is more in line with the aims of the course and, importantly, where computational tools are growing in new ways every day.

We have, I hope, seen how LLMs can help write code and facilitate things like taking our iNaturalist database and making an interactive map. But increasingly, there is even more that we can do with AI, and I want to spend some time exploring this.


## The data

The data we will use for this series of exercises was exported from Bioconductor's (an R package) [airway](https://bioconductor.org/packages/release/data/experiment/html/airway.html) dataset:

> This package provides a RangedSummarizedExperiment object of read counts in genes for an RNA-Seq experiment on four human airway smooth muscle cell lines treated with dexamethasone. Details on the gene model and read counting procedure are provided in the package vignette. The citation for the experiment is: Himes BE, Jiang X, Wagner P, Hu R, Wang Q, Klanderman B, Whitaker RM, Duan Q, Lasky-Su J, Nikolos C, Jester W, Johnson M, Panettieri R Jr, Tantisira KG, Weiss ST, Lu Q. 'RNA-Seq Transcriptome Profiling Identifies CRISPLD2 as a Glucocorticoid Responsive Gene that Modulates Cytokine Function in Airway Smooth Muscle Cells.' PLoS One. 2014 Jun 13;9(6):e99625. PMID: [24926665](https://pubmed.ncbi.nlm.nih.gov/24926665/). GEO: GSE52778.

### Himes *et al.* abstract

>Abstract
>
>Asthma is a chronic inflammatory respiratory disease that affects over 300 million people worldwide. Glucocorticoids are a mainstay therapy for asthma because they exert anti-inflammatory effects in multiple lung tissues, including the airway smooth muscle (ASM). However, the mechanism by which glucocorticoids suppress inflammation in ASM remains poorly understood. Using RNA-Seq, a high-throughput sequencing method, we characterized transcriptomic changes in four primary human ASM cell lines that were treated with dexamethasone--a potent synthetic glucocorticoid (1 µM for 18 hours). Based on a Benjamini-Hochberg corrected p-value <0.05, we identified 316 differentially expressed genes, including both well known (DUSP1, KLF15, PER1, TSC22D3) and less investigated (C7, CCDC69, CRISPLD2) glucocorticoid-responsive genes. CRISPLD2, which encodes a secreted protein previously implicated in lung development and endotoxin regulation, was found to have SNPs that were moderately associated with inhaled corticosteroid resistance and bronchodilator response among asthma patients in two previously conducted genome-wide association studies. Quantitative RT-PCR and Western blotting showed that dexamethasone treatment significantly increased CRISPLD2 mRNA and protein expression in ASM cells. CRISPLD2 expression was also induced by the inflammatory cytokine IL1β, and small interfering RNA-mediated knockdown of CRISPLD2 further increased IL1β-induced expression of IL6 and IL8. Our findings offer a comprehensive view of the effect of a glucocorticoid on the ASM transcriptome and identify CRISPLD2 as an asthma pharmacogenetics candidate gene that regulates anti-inflammatory effects of glucocorticoids in the ASM.

### Files

The files we have are in the `data` folder in this `airway` subfolder. The data were exported from Bioconductor with the `export_airway_data.r` script (generated by Claude). The `airway.db` file was generated with the `create_airway.db` script (generated by Claude).


**Summary of data files generated for the airway RNA‑seq example**  

| File | Description | Key Columns / Tables | Notes / Format |
|------|-------------|----------------------|----------------|
| **`airway_counts.csv`** | Gene‑level raw count matrix (genes × samples). Used for downstream quantification and differential expression. | • `gene_id` – Ensembl Gene ID (GRCh37/Ensembl 75)<br>• One column per sample (e.g., `SRR1039508`, `SRR1039509`, …) containing **raw integer read counts** | CSV, header row. No index column. Values are integers ≥ 0. |
| **`airway_samples.csv`** | Sample metadata (one row per library). Links run IDs to experimental conditions. | • `run_id` – SRA run accession (matches column names in `airway_counts.csv`)<br>• `SampleName` – descriptive sample label from the original study<br>• `cell` – cell‑line identifier (e.g., `N061011`, `N080611`, `N052611`, `N61311`)<br>• `dex` – treatment status: `"trt"` (dexamethasone 1 µM) or `"untrt"` (vehicle control) | CSV, header row. Can be joined to `airway_counts.csv` via `run_id`. |
| **`genes.csv`** | Gene annotation extracted from the `SummarizedExperiment` object (rowData). | • `gene_id` – Ensembl Gene ID (matches `airway_counts.csv` row names)<br>• `gene_name` – approved gene symbol (HGNC)<br>• `gene_biotype` – biotype (e.g., `protein_coding`, `lncRNA`, `processed_pseudogene`)<br>• Optional columns depending on Ensembl version: `seq_name` (chromosome), `gene_seq_start`, `gene_seq_end` (genomic coordinates) | CSV, header row. Provides a lookup table for gene symbols and biotypes; useful for filtering or reporting results. |
| **`messy_metadata.csv`** | Deliberately inconsistent sample sheet created for Session 2 (data‑cleaning exercise). Mimics real‑world metadata entry errors. | • `sample` – SRA run ID (same as `run_id` in other files)<br>• `condition` – free‑text description of treatment (e.g., `"Dex"`, `"dexamethasone 1uM"`, `"DEX treated"`, `"untrt"`, `"ctrl"` …)<br>• `donor_line` – cell‑line label with variations (case, spacing, prefix/suffix changes)<br>• `notes` – optional free‑text field (may be empty) | CSV, header row. Contains the same 8 samples as `airway_samples.csv` but with noisy values that students must map to standardized categories (`treatment`: dexamethasone / untreated; `cell_line`: exact cell‑line ID). |
| **`airway.db`** | SQLite database that stores the three tables above in a normalized, query‑friendly form. Enables SQL‑based exploration (Session 3) and read‑only tool use by the agent. | **Tables:**<br>• `samples(run_id TEXT PRIMARY KEY, SampleName TEXT, cell TEXT, dex TEXT)`<br>• `counts(gene_id TEXT, run_id TEXT, count INTEGER, PRIMARY KEY (gene_id, run_id))`<br>• `genes(gene_id TEXT PRIMARY KEY, gene_name TEXT, gene_biotype TEXT, …)`<br>**Indexes** (created automatically):<br>• `idx_counts_gene` on `counts(gene_id)`<br>• `idx_counts_run` on `counts(run_id)` | SQLite file (`data/airway.db`). Open in **read‑only** mode for agent tools (`sqlite3.connect("file:data/airway.db?mode=ro", uri=True)`). No writing allowed; all modifications must be done via the provided CSV files if needed. |

*All files are placed under the `data/` directory of the course repository.*

## Set up our notebook to work with NaviGator

## Step 1. Create a NaviGator API key

As we've seen API keys can be used to authenticate your scripts to an API endpoint. [NaviGator Toolkit](https://navigator.ai.ufl.edu/ai-services/navigator-ai-toolkit/) is UF's endpoint for LLMs. We'll use that for our exercises.

* Navigate to [https://api.navigator.ai.ufl.edu/ui/](https://api.navigator.ai.ufl.edu/ui/)
* Click on the "+ Create New Virtual Key" button.
    * Keep the Owned By as You
    * In needed, slect the "navigator-toolkit" team
    * Name the key, e.g. "bsc4452"
    * Leave the rest as default, and click "Create Key".

        ![Screenshot of creating an API key](images/create_api_key.png)

* **Be sure to copy the key before you click close!**
* Edit or create the `.env` file at the root of this repo (up one folder from this notebook, `../.env`)
    * Add a line: `OPENAI_API_KEY='sk-...'` where you past in your API key after the `=`.
* Now we can test that the API works!


In [18]:
import openai
import os
import json
from pathlib import Path
import os
from dotenv import load_dotenv
import pypdf

# Load environment variables from the ../.env file in the directory above the current working directory
load_dotenv("../.env")

# Retrieve the key
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
BASE_URL = "https://api.navigator.ai.ufl.edu/v1/" # This is the Navigator API base URL

# Get a list of available models
client = openai.OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=BASE_URL
)

response = client.models.list()
for model in response:
    print(model.id)


flux.2-klein
llama-3.3-70b-instruct
kokoro
gemma-4-31b-it
nemotron-3-super-120b-a12b
sfr-embedding-mistral
gpt-oss-20b
whisper-large-v3
mistral-small-3.1
nomic-embed-text-v1.5
meta-muse-glimmer-30b
nemotron-3-nano-30b-a3b
flux.1-dev
flux.1-schnell
gpt-oss-120b
granite-3.3-8b-instruct
medgemma-27b-it


### What we've done so far

The steps above should have walked you through creating your NaviGator API key, adding that key to your `.env` file, and tested connecting to NaviGator and retrieving the list of available models to use.

> ### Why .env?
> Remember that your API key can be used as you. Anyone with that key can use NaviGator as **you**.
>
> The .env file has already been added to the `.gitignore` file, so it won't be committed to GitHub. We are not adding the key to this notebook. All of this it to try and protect that key.

## Step 2. Send a test chat

Now that NaviGator is connected, let's chat!

We'll use the`nemotron-3-super-120b-a12b` model, but you could play with different models here if you want.

In [5]:
model = "nemotron-3-super-120b-a12b"  # Replace with the desired model ID

response = client.chat.completions.create(
    model=model,
    messages = [
        {
            "role": "user",
            "content": "Write a short poem about computin in biology."
        }
    ]
)

# Print nicely formatted message

print(response.choices[0].message.content)



Where silicon meets the double helix deep,  
Life’s source code hums in sequences vast—  
ATGC translates to 0101’s sleep,  
While algorithms parse the ancient past.  

No microscope alone can trace the thread  
That folds a protein, fights a rogue gene’s tide—  
But in the quiet hum where data’s spread,  
We watch life’s patterns turn, and open wide  
The doors to cure: a tumor’s silent flaw,  
A pandemic’s shadow, swiftly understood.  
Here, math and cell obey one quiet law—  
We learn to read, so we may learn to good.  

*(A short poem celebrating how computation decodes biology’s language—turning DNA’s alphabet into actionable insight for health and understanding.)*


In [7]:
# Let's look at the response object to see what other information is available
print(json.dumps(response.model_dump(), indent=4))

{
    "id": "chatcmpl-9472dd5f44e6c2ed",
    "choices": [
        {
            "finish_reason": "stop",
            "index": 0,
            "logprobs": null,
            "message": {
                "content": "\n\nWhere silicon meets the double helix deep,  \nLife\u2019s source code hums in sequences vast\u2014  \nATGC translates to 0101\u2019s sleep,  \nWhile algorithms parse the ancient past.  \n\nNo microscope alone can trace the thread  \nThat folds a protein, fights a rogue gene\u2019s tide\u2014  \nBut in the quiet hum where data\u2019s spread,  \nWe watch life\u2019s patterns turn, and open wide  \nThe doors to cure: a tumor\u2019s silent flaw,  \nA pandemic\u2019s shadow, swiftly understood.  \nHere, math and cell obey one quiet law\u2014  \nWe learn to read, so we may learn to good.  \n\n*(A short poem celebrating how computation decodes biology\u2019s language\u2014turning DNA\u2019s alphabet into actionable insight for health and understanding.)*",
                "refusal

In [8]:
# Ask the LLM about a gene:

gene_name = "TSC22D3"
response = client.chat.completions.create(
    model=model,
    messages = [
        {
            "role": "user",
            "content": f"Tell me about the gene {gene_name}."
        }
    ]
)       

print(response.choices[0].message.content)


Here's a comprehensive overview of the **TSC22D3 gene** (also widely known as **GILZ** - *Glucocorticoid-Induced Leucine Zipper*), covering its biology, function, regulation, and relevance to health and disease:

---

### **1. Basic Information**
*   **Official Symbol:** TSC22D3 (TSC22 Domain Family Member 3)
*   **Common Aliases:** GILZ, TSC22D3, DIP (Dexamethasone-Induced Protein), TSC22-3
*   **Gene Location:** Chromosome 20 (human: 20p13), mouse chromosome 2.
*   **Protein Product:** A ~20-24 kDa cytoplasmic/nuclear protein containing a **leucine zipper domain** (critical for dimerization and protein-protein interactions) and a conserved TSC22 domain. It belongs to the TSC22/DSC family of transcriptional regulators/modulators.
*   **Key Feature:** It is a **primary and robustly induced target gene** of glucocorticoid (GC) signaling, hence the name GILZ.

---

### **2. Regulation & Expression**
*   **Primary Inducer:** **Glucocorticoids (GCs)** (e.g., cortisol, dexamethasone) via t

In [12]:
# Not bad! Let's try a fake gene name to see how the model responds:
gene_name = "ZQX7"

response = client.chat.completions.create(
    model=model,
    messages = [
        {
            "role": "user",
            "content": f"Tell me about the gene {gene_name}."
        }
    ]
)       

print(response.choices[0].message.content)



After reviewing my knowledge base (current as of July 2024), **there is no widely recognized, well-characterized gene named "ZQX7" in major genomic databases** (such as NCBI Gene, HGNC, Ensembl, or UniProt) for humans, model organisms (mouse, rat, zebrafish, fruit fly, *C. elegans*, yeast), or commonly studied pathogens.

Here's a breakdown of why this might be the case and what you can do:

### Possible Explanations:
1.  **Typo or Misremembered Symbol:** Gene symbols are case-sensitive and often follow specific patterns. Common similar-looking symbols include:
    *   **ZFX:** A well-known gene (Zinc Finger Protein, X-linked) involved in transcription regulation, stem cell pluripotency, and development. Mutations are linked to developmental disorders.
    *   **ZBTB family** (e.g., ZBTB16, ZBTB20, ZBTB33): Zinc finger and BTB domain-containing genes, important in transcription, development, and cancer.
    *   **ZFHX family** (e.g., ZFHX3, ZFHX4): Zinc finger homeodomain genes, cruc

In [17]:
# Now, let's see how temperature affects the model's responses. We'll ask the same question with different temperature settings.
temperature = 0.5
response = client.chat.completions.create(
    model=model,
    messages = [
        {
            "role": "user",
            "content": f"Tell me about the gene {gene_name}."
        }
    ],
    temperature=temperature
)       

print(response.choices[0].message.content)



After checking major genomic databases (including **HGNC** [HUGO Gene Nomenclature Committee], **NCBI Gene**, **Ensembl**, and **UniProt**) as of my knowledge cutoff in July 2024, **there is no established, officially recognized gene named *ZQX7*** in humans, mice, rats, or other commonly studied model organisms.

Here's what this likely means and how to proceed:

### Possible Explanations:
1.  **Typo or Misremembered Name:**  
    Gene names can be easily confused (e.g., similar symbols like *ZFX*, *ZFY*, *ZBTB7A*, *ZNF7*, *ZSC7*, or *ZXDA*). Double-check the spelling:
    - Did you mean **ZFX** (Zinc Finger Protein, X-linked)?  
    - Or **ZBTB7** (Zinc Finger and BTB Domain Containing 7)?  
    - Or perhaps **ZNF7** (Zinc Finger Protein 7)?  
    - Lab-specific aliases or older names (e.g., some genes had temporary designations before HGNC approval) could also cause confusion.

2.  **Very Recent or Niche Discovery:**  
    It *could* be a gene identified in a *very recent* study (

## Step 3. Paper analysis

Let's see how an LLM can be used to get information about the study from the paper.

The 